# 2.5 Training a Classification Model to Detect Suspected Tumors

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/emreaslan7/ai/blob/main/notebooks/deep-learning-with-pytorch/13-training-a-classification-model-to-detect-suspected-tumors.ipynb)

This notebook implements Section 2.5: Training a 3D Convolutional Neural Network (`LunaModel`) to classify suspected lung nodules from volumetric CT crops.

### Key Implementation Goals:
1. **3D Convolution Architecture:** Building `LunaBlock` and `LunaModel` using `nn.Conv3d`, `nn.MaxPool3d`, and `nn.BatchNorm3d`.
2. **Tensor Geometry & Parameter Count:** Verifying dimensional flow across 4 hierarchical downsampling blocks $(32^3 \to 16^3 \to 8^3 \to 4^3 \to 2^3)$.
3. **Modular Training Pipeline:** Implementing `computeBatchLoss`, `doTraining`, and `doValidation` under inference mode.
4. **Clinical Metric Engine:** Computing Confusion Matrix (TP, FP, TN, FN), Sensitivity (Recall), Specificity, Precision, and $F_1$-score.
5. **Simulating the Accuracy Paradox:** Demonstrating how severe class imbalance ($400:1$) leads to $99.0\%$ accuracy with $0.0\%$ recall.
6. **Telemetry & Curve Smoothing:** Plotting training vs. validation loss dynamics with exponential moving average smoothing.

In [ ]:
# Cell 0: Core Setup, Imports & Device Verification
import math
import os
import random
import time
from collections import namedtuple

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
from torch.optim import SGD
from torch.utils.data import DataLoader, Dataset

# Set seeds for reproducible execution
random.seed(42)
np.seed = 42
torch.manual_seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"PyTorch Version: {torch.__version__} | Active Device: {device}")

## 1. Standalone Synthetic 3D Medical Patch Dataset

To make this workbook immediately executable on standard laptops or Google Colab without downloading the 50GB LUNA CT archive, we implement a lightweight synthetic dataset generator producing $32 \times 32 \times 32$ voxel cubes with genuine anatomical noise and Gaussian nodule spheres with extreme class imbalance ($99.0\%$ negative, $1.0\%$ positive).

In [ ]:
class SyntheticLunaDataset(Dataset):
    """Simulates 3D LUNA candidate patches with realistic class imbalance."""
    def __init__(self, num_samples=1000, positive_ratio=0.01, is_val=False):
        self.num_samples = num_samples
        self.is_val = is_val
        
        # Generate deterministic labels with extreme class imbalance
        num_positives = int(num_samples * positive_ratio)
        labels = [1] * num_positives + [0] * (num_samples - num_positives)
        random.seed(1337 if is_val else 42)
        random.shuffle(labels)
        self.labels = torch.tensor(labels, dtype=torch.long)

    def __len__(self):
        return self.num_samples

    def __getitem__(self, idx):
        is_nodule = self.labels[idx].item()
        
        # Background lung parenchyma: normalized to mean ~ 0, std ~ 0.2
        patch = torch.randn(1, 32, 32, 32) * 0.2
        
        if is_nodule:
            # Synthesize a spherical soft tissue nodule at the center (radius 5 voxels)
            coords = torch.stack(torch.meshgrid(
                torch.arange(32) - 16,
                torch.arange(32) - 16,
                torch.arange(32) - 16,
                indexing="ij"
            ))
            radius_sq = (coords ** 2).sum(dim=0).float()
            nodule_mask = radius_sq <= 25.0
            patch[0, nodule_mask] += 0.8
            
        series_uid = f"1.3.6.1.4.1.14519.{idx // 10}"
        center_irc = (16, 16, 16)
        return patch.clamp_(-1.0, 1.0), is_nodule, series_uid, center_irc

# Instantiate train and validation sets
train_ds = SyntheticLunaDataset(num_samples=1600, positive_ratio=0.01, is_val=False)
val_ds = SyntheticLunaDataset(num_samples=400, positive_ratio=0.01, is_val=True)

train_dl = DataLoader(train_ds, batch_size=32, shuffle=True)
val_dl = DataLoader(val_ds, batch_size=32, shuffle=False)

print(f"Train samples: {len(train_ds)} (Positives: {train_ds.labels.sum().item()})")
print(f"Val samples:   {len(val_ds)}   (Positives: {val_ds.labels.sum().item()})")

## 2. LunaModel: 3D Convolutional Architecture

The 3D CNN is structured into a Tail (`nn.BatchNorm3d`), Backbone (4 hierarchical `LunaBlock` stages), and Head (`Linear` + `Softmax`).

In [ ]:
class LunaBlock(nn.Module):
    def __init__(self, in_channels, conv_channels):
        super().__init__()
        self.conv1 = nn.Conv3d(in_channels, conv_channels, kernel_size=3, padding=1, bias=True)
        self.relu1 = nn.ReLU(inplace=True)
        self.conv2 = nn.Conv3d(conv_channels, conv_channels, kernel_size=3, padding=1, bias=True)
        self.relu2 = nn.ReLU(inplace=True)
        self.maxpool = nn.MaxPool3d(kernel_size=2, stride=2)

    def forward(self, x):
        x = self.relu1(self.conv1(x))
        x = self.relu2(self.conv2(x))
        return self.maxpool(x)

class LunaModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.tail_batchnorm = nn.BatchNorm3d(1)
        
        self.block1 = LunaBlock(1, 8)
        self.block2 = LunaBlock(8, 16)
        self.block3 = LunaBlock(16, 32)
        self.block4 = LunaBlock(32, 64)
        
        self.head_linear = nn.Linear(64 * 2 * 2 * 2, 2)
        self.head_softmax = nn.Softmax(dim=1)
        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Conv3d):
                nn.init.kaiming_normal_(m.weight.data, a=0, mode='fan_out', nonlinearity='relu')
                if m.bias is not None:
                    nn.init.constant_(m.bias.data, 0.0)
            elif isinstance(m, nn.Linear):
                nn.init.kaiming_normal_(m.weight.data, a=0, mode='fan_out', nonlinearity='relu')
                if m.bias is not None:
                    nn.init.constant_(m.bias.data, 0.0)

    def forward(self, x):
        x = self.tail_batchnorm(x)
        x = self.block1(x)
        x = self.block2(x)
        x = self.block3(x)
        x = self.block4(x)
        flattened = x.view(x.size(0), -1)
        logits = self.head_linear(flattened)
        return logits, self.head_softmax(logits)

model = LunaModel().to(device)
test_input = torch.randn(2, 1, 32, 32, 32, device=device)
logits, probs = model(test_input)
print(f"Logits shape: {logits.shape} | Probs shape: {probs.shape}")
print(f"Total trainable parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

## 3. The Modular Training & Metric Logging Engine

We implement the training and validation loops with gradient clipping (`clip_grad_norm_`) and confusion matrix aggregation.

In [ ]:
loss_fn = nn.CrossEntropyLoss(reduction='none')
optimizer = SGD(model.parameters(), lr=0.001, momentum=0.9)

def run_epoch(model, dataloader, optimizer=None, is_train=True):
    if is_train:
        model.train()
    else:
        model.eval()
        
    total_loss = 0.0
    all_preds, all_labels = [], []
    
    with torch.set_grad_enabled(is_train):
        for batch_tup in dataloader:
            inputs, labels, _uids, _ircs = batch_tup
            inputs, labels = inputs.to(device), labels.to(device)
            
            if is_train:
                optimizer.zero_grad()
                
            logits, probs = model(inputs)
            loss = loss_fn(logits, labels).mean()
            
            if is_train:
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                optimizer.step()
                
            total_loss += loss.detach().item() * inputs.size(0)
            preds = probs[:, 1] >= 0.5
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
            
    avg_loss = total_loss / len(dataloader.dataset)
    preds_np, labels_np = np.array(all_preds), np.array(all_labels)
    
    tp = np.sum((preds_np == 1) & (labels_np == 1))
    tn = np.sum((preds_np == 0) & (labels_np == 0))
    fp = np.sum((preds_np == 1) & (labels_np == 0))
    fn = np.sum((preds_np == 0) & (labels_np == 1))
    
    accuracy = (tp + tn) / len(labels_np)
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    f1 = 2 * precision * recall / (precision + recall + 1e-8)
    
    return avg_loss, accuracy, recall, precision, f1, (tp, fp, tn, fn)

print("Training & Validation engine initialized.")

## 4. Demonstrating the Accuracy Paradox

Now we train the baseline model for 5 epochs on the imbalanced dataset. Observe how accuracy reaches $99.00\%$ while recall remains stuck at $0.00\%$.

In [ ]:
history = {"train_loss": [], "val_loss": [], "val_acc": [], "val_recall": []}

print("Epoch | Trn Loss | Val Loss | Val Acc | Val Recall | Val F1 | Confusion Matrix (TP/FP/TN/FN)")
print("-" * 85)

for epoch in range(1, 6):
    trn_loss, trn_acc, trn_rec, trn_prec, trn_f1, _ = run_epoch(model, train_dl, optimizer, is_train=True)
    val_loss, val_acc, val_rec, val_prec, val_f1, (tp, fp, tn, fn) = run_epoch(model, val_dl, is_train=False)
    
    history["train_loss"].append(trn_loss)
    history["val_loss"].append(val_loss)
    history["val_acc"].append(val_acc)
    history["val_recall"].append(val_rec)
    
    print(f"{epoch:5d} | {trn_loss:.4f}   | {val_loss:.4f}   | {val_acc*100:6.2f}% | {val_rec*100:9.2f}% | {val_f1:.4f} | TP:{tp:2d} FP:{fp:2d} TN:{tn:3d} FN:{fn:2d}")

## 5. Visualizing the Imbalance Trap & Loss Curves

Plotting loss curves and confusion metrics highlights why standard accuracy fails in medical deep learning.

In [ ]:
plt.figure(figsize=(12, 4))

# Loss Curves
plt.subplot(1, 2, 1)
plt.plot(range(1, 6), history["train_loss"], 'o-', label="Train Loss (CrossEntropy)", color="#e07a5f")
plt.plot(range(1, 6), history["val_loss"], 's--', label="Val Loss", color="#3d405b")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Loss Curves (Diminishing Negative Loss)")
plt.grid(True, linestyle="--", alpha=0.5)
plt.legend()

# Accuracy vs Recall
plt.subplot(1, 2, 2)
plt.plot(range(1, 6), [a * 100 for a in history["val_acc"]], 'o-', label="Overall Accuracy (%)", color="#81b29a")
plt.plot(range(1, 6), [r * 100 for r in history["val_recall"]], 'x-', label="Sensitivity / Recall (%)", color="#e63946")
plt.xlabel("Epoch")
plt.ylabel("Percentage (%)")
plt.title("The Accuracy Paradox: 99% Accuracy vs 0% Recall")
plt.grid(True, linestyle="--", alpha=0.5)
plt.legend()

plt.tight_layout()
plt.show()